# Automated Claim Triage: an improved prompt chain

Companion notebook for **lesson 4.4** of the study notes. Three stages turn a free-text auto-insurance claim (FNOL) into a queue decision:

1. **Extract** claim info with the LLM → Gate 1 (Pydantic)
2. **Assess** severity and cost with the LLM → Gate 2 (Pydantic + cost range)
3. **Route** with plain code: the rules are fixed, so no LLM call

Failures retry with the gate's error as feedback; after the retry limit the claim goes to `needs_review`, so nothing is dropped.

**Running it:** with no API key, the demo cells use scripted replies. To use a real model, add `GEMINI_API_KEY` or `OPENAI_API_KEY` in Colab's **Secrets** panel (key icon, left sidebar) and switch on **Notebook access**.

**Run the setup cell first**: it loads the helper and reports which settings it found.

In [1]:
# Setup: load the shared get_completion() helper, from the repo or (in Colab) from GitHub
import os, sys, urllib.request
for path in ["../../shared", "."]:
    if os.path.exists(os.path.join(path, "llm.py")):
        sys.path.insert(0, path)
        break
else:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/deresegetachew/study-notes/main/udacity-masters-agentic-ai/code/shared/llm.py", "llm.py")
    sys.path.insert(0, ".")

from llm import get_completion, set_mock_replies, check_setup
check_setup()   # shows which LLM settings were found, and why any are missing

LLM: mock (scripted replies only)
  · LLM_PROVIDER    not found: not set as an environment variable
  · GEMINI_API_KEY  not found: not set as an environment variable
  · OPENAI_API_KEY  not found: not set as an environment variable
  · LLM_MODEL       not found: not set as an environment variable


'LLM: mock (scripted replies only)\n  · LLM_PROVIDER    not found: not set as an environment variable\n  · GEMINI_API_KEY  not found: not set as an environment variable\n  · OPENAI_API_KEY  not found: not set as an environment variable\n  · LLM_MODEL       not found: not set as an environment variable'

## 1 · The data contracts

In [2]:
import json, re
from typing import List, Literal
from pydantic import BaseModel, Field, ValidationError

class ClaimInformation(BaseModel):                     # Stage I output
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[Literal["windshield", "front", "rear", "side", "roof", "hood", "door",
                              "bumper", "fender", "quarter panel", "trunk", "glass"]] = Field(..., min_length=1)

class SeverityAssessment(BaseModel):                   # Stage II output
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)

## 2 · Prompts (from the course exercise)

In [3]:
info_extraction_system_prompt = """
You are an auto insurance claim processing assistant. Extract key information from the
First Notice of Loss (FNOL) report as a JSON object with keys: claim_id, name, vehicle,
loss_desc (concise), damage_area (list; only: windshield, front, rear, side, roof, hood,
door, bumper, fender, quarter panel, trunk, glass). Only respond with the JSON object.
"""

severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Apply these carrier heuristics:
- Minor: small dents, scratches, glass chips ($100-$1,000)
- Moderate: single panel, bumper replacement, door damage ($1,000-$5,000)
- Major: structural, multiple panels, engine/drivetrain, total-loss candidates ($5,000-$50,000)
Respond only with a JSON object: {"severity": "Minor"|"Moderate"|"Major", "est_cost": number}.
"""

## 3 · Gate checks

In [4]:
COST_RANGES = {"Minor": (100, 1_000), "Moderate": (1_000, 5_000), "Major": (5_000, 50_000)}

def gate1(raw: str) -> ClaimInformation:
    """Shape check: valid JSON with the right fields and allowed damage areas."""
    try:
        return ClaimInformation.model_validate_json(raw)
    except ValidationError as e:
        raise ValueError(str(e))

def gate2(raw: str) -> SeverityAssessment:
    """Shape check + logic check: the cost must fit the chosen severity."""
    try:
        sev = SeverityAssessment.model_validate_json(raw)
    except ValidationError as e:
        raise ValueError(str(e))
    low, high = COST_RANGES[sev.severity]
    if not low <= sev.est_cost <= high:
        raise ValueError(f"{sev.severity} damage must cost {low:,}-{high:,} USD, got {sev.est_cost:,.0f}")
    return sev

## 4 · Routing in plain code

In [5]:
GLASS_ONLY = {"windshield", "glass"}

def route(claim: ClaimInformation, sev: SeverityAssessment) -> str:
    """Stage III: the routing rules are fixed, so no LLM call is needed."""
    if sev.severity == "Major":
        return "total_loss"
    if sev.severity == "Moderate":
        return "material_damage"
    if set(claim.damage_area) <= GLASS_ONLY:
        return "glass"
    return "fast_track"

def _claim(areas):
    return ClaimInformation(claim_id="C1", name="Al", vehicle="Car", loss_desc="something happened", damage_area=areas)

for areas, severity, cost in [(["windshield"], "Minor", 150), (["windshield", "door"], "Minor", 300),
                              (["door"], "Moderate", 2000), (["front"], "Major", 15000)]:
    print(f"{severity:8} {str(areas):24} -> {route(_claim(areas), SeverityAssessment(severity=severity, est_cost=cost))}")

Minor    ['windshield']           -> glass
Minor    ['windshield', 'door']   -> fast_track
Moderate ['door']                 -> material_damage
Major    ['front']                -> total_loss


## 5 · The chain: retries with feedback, then human review

In [6]:
class GateFailed(Exception):
    pass

def strip_fences(text: str) -> str:
    """Models often wrap JSON in a Markdown fence; remove it before parsing."""
    return re.sub(r"^\s*`{3}(?:json)?\s*\n|\n?`{3}\s*$", "", text.strip())

def call_with_gate(system_prompt: str, user_content: str, gate, max_attempts: int = 3):
    """One chain step: call the LLM, run the gate, retry with the gate's error as feedback."""
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}]
    for attempt in range(1, max_attempts + 1):
        raw = strip_fences(get_completion(messages=messages, temperature=0))
        try:
            return gate(raw)
        except ValueError as e:
            if attempt == max_attempts:
                raise GateFailed(f"{gate.__name__} failed {max_attempts}x: {e}")
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your answer failed validation:\n{e}\nReturn corrected JSON only."}]

def triage(fnol_text: str) -> dict:
    """Never drops a claim: anything that fails a gate goes to human review."""
    try:
        claim = call_with_gate(info_extraction_system_prompt, fnol_text, gate1)
        sev = call_with_gate(severity_assessment_system_prompt, claim.model_dump_json(), gate2)
        return {"claim_id": claim.claim_id, "severity": sev.severity, "est_cost": sev.est_cost,
                "queue": route(claim, sev), "status": "auto"}
    except GateFailed as e:
        return {"claim_id": None, "queue": "needs_review", "status": "human", "reason": str(e)[:120]}

## 6 · Try it: three scripted scenarios

`set_mock_replies(...)` scripts exactly what the "LLM" returns, so each scenario is reproducible with or without an API key.

In [7]:
fnol = """Claim ID: C001
Customer: John Smith
Vehicle: 2018 Toyota Camry
Incident: A rock hit my windshield and caused a small chip. No other damage."""

claim_json = ('{"claim_id": "C001", "name": "John Smith", "vehicle": "2018 Toyota Camry", '
              '"loss_desc": "Rock chipped the windshield.", "damage_area": ["windshield"]}')

# A. Happy path — the extraction even comes back wrapped in a Markdown fence
set_mock_replies("`" * 3 + "json\n" + claim_json + "\n" + "`" * 3, '{"severity": "Minor", "est_cost": 150}')
print("A.", triage(fnol))

# B. Gate 2 fails (cost too high for Minor), the retry with feedback fixes it
set_mock_replies(claim_json, '{"severity": "Minor", "est_cost": 3000}', '{"severity": "Minor", "est_cost": 300}')
print("B.", triage(fnol))

# C. Extraction keeps failing → human review, nothing dropped
set_mock_replies("not json", "still not json", "nope")
print("C.", triage(fnol))

A. {'claim_id': 'C001', 'severity': 'Minor', 'est_cost': 150.0, 'queue': 'glass', 'status': 'auto'}
B. {'claim_id': 'C001', 'severity': 'Minor', 'est_cost': 300.0, 'queue': 'glass', 'status': 'auto'}
C. {'claim_id': None, 'queue': 'needs_review', 'status': 'human', 'reason': 'gate1 failed 3x: 1 validation error for ClaimInformation\n  Invalid JSON: expected ident at line 1 column 2 [type=json_in'}


## 7 · With a real model

Add `GEMINI_API_KEY` or `OPENAI_API_KEY` in Colab's **Secrets** panel (key icon, left sidebar), switch on **Notebook access** for it, re-run the setup cell, then run this cell on the course's sample claims. Without a key it stops with instructions.

In [8]:
check_setup(require_key=True)   # stops here with instructions if no API key is set

samples = [
    "Claim ID: C002\nCustomer: Sarah Johnson\nVehicle: 2020 Honda Civic\nIncident: Parked car was hit; "
    "rear bumper dented and taillight broken.",
    "Claim ID: C003\nCustomer: Michael Rodriguez\nVehicle: 2022 Ford F-150\nIncident: Serious collision; "
    "hood, bumper, radiator and engine damaged, airbags deployed, not drivable.",
]
for s in samples:
    print(triage(s))

LLM: mock (scripted replies only)
  · LLM_PROVIDER    not found: not set as an environment variable
  · GEMINI_API_KEY  not found: not set as an environment variable
  · OPENAI_API_KEY  not found: not set as an environment variable
  · LLM_MODEL       not found: not set as an environment variable


RuntimeError: This cell needs a real LLM, but no usable API key was found.
In Colab: click the key icon (Secrets) in the left sidebar, add GEMINI_API_KEY or
OPENAI_API_KEY, switch on 'Notebook access' for it, then re-run the setup cell.
Locally: export GEMINI_API_KEY=... (or OPENAI_API_KEY) before starting Jupyter.